### PLINK & GWAS

- QC

https://plink.readthedocs.io/en/latest/GWAS/

- Regression

https://www.cog-genomics.org/plink/2.0/tutorials/qwas1

- Paper

https://pubmed.ncbi.nlm.nih.gov/23756892/

Rentería ME, Cortes A, Medland SE. Using PLINK for Genome-Wide Association Studies (GWAS) and data analysis. Methods Mol Biol. 2013;1019:193-213. doi: 10.1007/978-1-62703-447-0_8. PMID: 23756892.

### BED format


https://genome.ucsc.edu/FAQ/FAQformat.html

https://genome.ucsc.edu/FAQ/FAQformat.html#format1

BED (Browser Extensible Data) format provides a flexible way to define the data lines that are displayed in an annotation track. BED lines have three required fields and nine additional optional fields. The number of fields per line must be consistent throughout any single set of data in an annotation track. The order of the optional fields is binding: lower-numbered fields must always be populated if higher-numbered fields are used.

BED information should not be mixed as explained above (BED3 should not be mixed with BED4), rather additional column information must be filled for consistency, for example with a "." in some circumstances, if the field content is to be empty. BED fields in custom tracks can be whitespace-delimited or tab-delimited. Only some variations of BED types, such as bedDetail, require a tab character delimitation for the detail columns.


```Bash
track name=pairedReads description="Clone Paired Reads" useScore=1
chr22 1000 5000 cloneA 960 + 1000 5000 0 2 567,488, 0,3512
chr22 2000 6000 cloneB 900 - 2000 6000 0 2 433,399, 0,3601
```

The first three required BED fields are:

1. chrom - The name of the chromosome (e.g. chr3, chrY, chr2_random) or scaffold (e.g. scaffold10671). Many assemblies also support several different chromosome aliases (e.g. '1' or 'NC_000001.11' in place of 'chr1').
2. chromStart - The starting position of the feature in the chromosome or scaffold. The first base in a chromosome is numbered 0.
3. chromEnd - The ending position of the feature in the chromosome or scaffold. The chromEnd base is not included in the display of the feature, however, the number in position format will be represented. For example, the first 100 bases of chromosome 1 are defined as chrom=1, chromStart=0, chromEnd=100, and span the bases numbered 0-99 in our software (not 0-100), but will represent the position notation chr1:1-100. Read more here.
chromStart and chromEnd can be identical, creating a feature of length 0, commonly used for insertions. For example, use chromStart=0, chromEnd=0 to represent an insertion before the first nucleotide of a chromosome.


The 9 additional optional BED fields are:

4. name - Defines the name of the BED line. This label is displayed to the left of the BED line in the Genome Browser window when the track is open to full display mode or directly to the left of the item in pack mode.
5. score - A score between 0 and 1000. If the track line useScore attribute is set to 1 for this annotation data set, the score value will determine the level of gray in which this feature is displayed (higher numbers = darker gray). This table shows the Genome Browser's translation of BED score values into shades of gray (see link)
score in range  	≤ 166	167-277	278-388	389-499	500-611	612-722	723-833	834-944	≥ 945

6. strand - Defines the strand. Either "." (=no strand) or "+" or "-".
7. thickStart - The starting position at which the feature is drawn thickly (for example, the start codon in gene displays). When there is no thick part, thickStart and thickEnd are usually set to the chromStart position.
8. thickEnd - The ending position at which the feature is drawn thickly (for example the stop codon in gene displays).
9. itemRgb - An RGB value of the form R,G,B (e.g. 255,0,0). If the track line itemRgb attribute is set to "On", this RGB value will determine the display color of the data contained in this BED line. NOTE: It is recommended that a simple color scheme (eight colors or less) be used with this attribute to avoid overwhelming the color resources of the Genome Browser and your Internet browser.
10. blockCount - The number of blocks (exons) in the BED line.
11. blockSizes - A comma-separated list of the block sizes. The number of items in this list should correspond to blockCount.
12. blockStarts - A comma-separated list of block starts. All of the blockStart positions should be calculated relative to chromStart. The number of items in this list should correspond to blockCount.

In BED files with block definitions, the first blockStart value must be 0, so that the first block begins at chromStart. Similarly, the final blockStart position plus the final blockSize value must equal chromEnd. Blocks may not overlap.

### Commands

- convert to plink format.

```Bash
$ plink2 --vcf data.vcf --set-missing-var-ids @_#_\$r_\$a --new-id-max-allele-len 100 --make-bed --out data
or
$ plink2 --vcf data.vcf.gz --set-missing-var-ids @_#_\$r_\$a --new-id-max-allele-len 100 --make-bed --out data
```

> @ : chromosome  
> \# : position  
> \$r : reference allele  
> \$a : alt allele  


- Remove missing variants and samples alternately from 0.1 to 0.01.

```Bash
$ plink2 --bfile raw_data --geno (0.1~0.01) --make-bed --out ft_missing
$ plink2 --bfile ft_missing --mind (0.1~0.01) --make-bed --out ft_missing
```


## can you explain " trait has 80% heritability and 50% prevalence"


Both numbers come from the single GCTA command the tutorial uses to invent its phenotype:

```
gcta --bfile 1KG.EAS... --simu-cc 250 254 \
     --simu-causal-loci causal.snplist --simu-hsq 0.8 --simu-k 0.5
```

`--simu-hsq 0.8` is the heritability, `--simu-k 0.5` the prevalence.

### Prevalence (K = 0.5)

Prevalence is simply the fraction of the population that has the disease. K = 0.5 means half of everyone is a case.

It enters the simulation through the **liability threshold model**, the standard way of treating a yes/no disease as if it came from a continuous cause. Each person is given an unobserved "liability" — total genetic plus environmental burden — drawn from a standard normal distribution. Everyone above a threshold T is a case, everyone below is a control. The prevalence is what sets T:

| Prevalence | Threshold T | Real-world example |
|---|---|---|
| 50% | 0.00 SD | the tutorial's simulated trait |
| 10% | 1.28 SD | Alzheimer's in the over-65 population |
| 5% | 1.64 SD | coronary artery disease, roughly |
| 1.7% | 2.12 SD | lifetime pancreatic cancer risk |

So K = 0.5 puts the cutoff exactly at the population mean. A real disease sits far out in the tail. At K = 0.5 you are not modelling a disease at all — you are modelling something like "above-average height", a trait half the population has.

This matters practically because the threshold sits where the liability distribution is densest. Cases and controls then differ maximally in average liability per person sampled, and a balanced 250-vs-254 design is as statistically efficient as a case/control study can be. Push the threshold out to 2 SD and most of your controls carry liability nowhere near the boundary, so each sample tells you much less.

## Heritability (h² = 0.8)

Heritability is the **proportion of variance in the trait — here, in liability — explained by additive genetic effects**. h² = 0.8 means 80% of the variance in liability comes from the genotypes, and 20% from everything else.

Three things it is not:
- It is not "this disease is 80% genetic" for any individual. It is a ratio of vari0/1 scale. The two differ, and converting between them requires knowing K — which is one more reason prevalence has to be specified.

## Why the combination makes the exercise unrealistically easy

The decisive detail is how the 80% is distributed. The causal file holds just **five variants**, each assigned an effect size of 3, picked at random from the genome by the commented-out line in the script:

```
shuf ...bim | head -n 5 | awk '{print $2, 3}' > causal.snplist
```

So five SNPs carry 80% of liability variance — roughly 16% each. For comparison, a typical real GWAS hit for a complex disease explains well under 0.1% of variance, and the trait's heritability is spread across thousands or tens of thousands of variants. Even APOE, by far the largances in a particular population with a particular range of environments. Change the environment and the same genes give a different h².
- GCTA simulates only **additive** effects, so this is narrow-sense h², excluding dominance and epistasis.
- It is on the **liability scale** here, not the observed est common-variant effect in late-onset Alzheimer's, accounts for only 
nt of liability.

That is why 504 individuals suffice to produce a clean Manhattan plot in the tutorial. A real study with five such variants would have been solved in the 1990s by linkage analysis, without any need for a GWAS.

It is a sensible teaching choice — you want the signal visible so the mechanics of QC, association and plotting are what you're learning. But it is worth knowing that the plot you get is not what real data looks like.

Your `adgwas` data shows the contrast: APOE e4 carriers are 68.8% of cases versus 21.3% of controls, which is a genuinely strong genetic effect by real-world standards, and it still comes from a locus explaining only a small share of liability in a cohort of 364. Expect one region to stand out and the rest of the genome to look like noise, because with real heritability spread thinly, at that sample size it mostly is.